# P01：金融数据获取

本Notebook用于下载以下数据：
1. 10只A股股票的后复权日度行情数据
2. 市场指数数据（沪深300、中证500）
3. 宏观经济指标（CPI、M2）
4. 财务指标数据

数据来源：akshare（免费，无需注册）

## 1. 环境准备

In [ ]:
# 导入必要的库
import os
import time
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import akshare as ak

# 打印akshare版本
print(f"akshare版本: {ak.__version__}")
print(f"pandas版本: {pd.__version__}")

In [ ]:
# 定义股票列表（覆盖7个行业，共10只股票）
# 选择理由：选择各行业龙头企业，代表性强，流动性好
stock_list = [
    # 银行行业
    {"code": "600036", "name": "招商银行", "industry": "银行"},
    {"code": "601398", "name": "工商银行", "industry": "银行"},
    # 汽车行业
    {"code": "002594", "name": "比亚迪", "industry": "汽车"},
    {"code": "600104", "name": "上汽集团", "industry": "汽车"},
    # 房地产行业
    {"code": "000002", "name": "万科A", "industry": "房地产"},
    # 白酒行业
    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "000858", "name": "五粮液", "industry": "白酒"},
    # 能源行业
    {"code": "601857", "name": "中国石油", "industry": "能源"},
    # 通讯行业
    {"code": "000063", "name": "中兴通讯", "industry": "通讯"},
    # 物流行业
    {"code": "002352", "name": "顺丰控股", "industry": "物流"},
]

# 转换为DataFrame便于查看
stock_df = pd.DataFrame(stock_list)
print("股票列表：")
display(stock_df)

In [ ]:
# 定义时间范围
start_date = "20200101"
end_date = datetime.now().strftime("%Y%m%d")
print(f"数据时间范围: {start_date} 至 {end_date}")

# 定义数据存储路径
data_dir = "data"
stock_dir = os.path.join(data_dir, "stock")
index_dir = os.path.join(data_dir, "index")
macro_dir = os.path.join(data_dir, "macro")
finance_dir = os.path.join(data_dir, "finance")
log_file = "download_log.txt"

## 2. 定义下载日志函数

In [ ]:
def write_log(log_file, status, data_name, message=""):
    """
    记录下载日志到文件
    
    Parameters:
    -----------
    log_file : str - 日志文件路径
    status : str - 状态（SUCCESS/FAILED）
    data_name : str - 数据名称
    message : str - 附加信息
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if status == "SUCCESS":
        log_entry = f"[{timestamp}] SUCCESS  {data_name}  {message}\n"
    else:
        log_entry = f"[{timestamp}] FAILED   {data_name}  Error: {message}\n"
    
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(log_entry)
    print(log_entry.strip())

## 3. 下载股票数据

In [ ]:
def download_stock_data(stock_code, stock_name, start_date, end_date, save_dir):
    """
    下载单只股票的后复权日度行情数据
    
    Parameters:
    -----------
    stock_code : str - 股票代码
    stock_name : str - 股票名称
    start_date : str - 开始日期（格式：YYYYMMDD）
    end_date : str - 结束日期（格式：YYYYMMDD）
    save_dir : str - 保存目录
    
    Returns:
    --------
    bool - 是否下载成功
    """
    try:
        # 使用akshare获取后复权数据
        # adjust="hfq" 表示后复权
        df = ak.stock_zh_a_hist(
            symbol=stock_code,
            period="daily",
            start_date=start_date,
            end_date=end_date,
            adjust="hfq"  # 后复权
        )
        
        if df is None or len(df) == 0:
            return False, "No data returned"
        
        # 重命名列（akshare返回的列名）
        # 原始列名：日期、开盘、收盘、最高、最低、成交量、成交额、振幅、涨跌幅、涨跌额、换手率
        column_mapping = {
            "日期": "date",
            "开盘": "open",
            "收盘": "close",
            "最高": "high",
            "最低": "low",
            "成交量": "volume",
            "成交额": "amount"
        }
        
        # 选择需要的列并重命名
        df_selected = df[["日期", "开盘", "收盘", "最高", "最低", "成交量", "成交额"]].copy()
        df_selected.columns = ["date", "open", "close", "high", "low", "volume", "amount"]
        
        # 保存为CSV
        save_path = os.path.join(save_dir, f"stock_{stock_code}.csv")
        df_selected.to_csv(save_path, index=False, encoding="utf-8")
        
        return True, f"shape=({len(df_selected)}, {len(df_selected.columns)})"
        
    except Exception as e:
        return False, str(e)

In [ ]:
# 清空日志文件
with open(log_file, "w", encoding="utf-8") as f:
    f.write("")

print("开始下载股票数据...")
print("="*60)

success_count = 0
fail_count = 0

for stock in stock_list:
    code = stock["code"]
    name = stock["name"]
    
    print(f"\n正在下载: {name} ({code})...")
    
    success, msg = download_stock_data(code, name, start_date, end_date, stock_dir)
    
    if success:
        write_log(log_file, "SUCCESS", f"stock_{code}", msg)
        success_count += 1
    else:
        write_log(log_file, "FAILED", f"stock_{code}", msg)
        fail_count += 1
    
    # 添加延迟，避免请求过快
    time.sleep(0.5)

print("\n" + "="*60)
print(f"股票数据下载完成！成功: {success_count}, 失败: {fail_count}")

## 4. 下载市场指数数据

In [ ]:
def download_index_data(index_code, index_name, start_date, end_date, save_dir):
    """
    下载指数数据
    
    Parameters:
    -----------
    index_code : str - 指数代码（如"000300"为沪深300）
    index_name : str - 指数名称
    start_date : str - 开始日期
    end_date : str - 结束日期
    save_dir : str - 保存目录
    """
    try:
        # 使用akshare获取指数数据
        df = ak.stock_zh_index_daily(symbol=f"sh{index_code}")
        
        if df is None or len(df) == 0:
            return False, "No data returned"
        
        # 筛选日期范围
        df['date'] = pd.to_datetime(df['date'])
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)
        df = df[(df['date'] >= start_dt) & (df['date'] <= end_dt)]
        
        # 选择需要的列
        df_selected = df[['date', 'open', 'close', 'high', 'low', 'volume']].copy()
        df_selected['date'] = df_selected['date'].dt.strftime('%Y-%m-%d')
        
        # 保存
        save_path = os.path.join(save_dir, f"index_{index_code}.csv")
        df_selected.to_csv(save_path, index=False, encoding="utf-8")
        
        return True, f"shape=({len(df_selected)}, {len(df_selected.columns)})"
        
    except Exception as e:
        return False, str(e)

In [ ]:
# 定义指数列表
# 沪深300作为CAPM市场基准（必选）
# 中证500作为自选指数（代表中小盘股票）
index_list = [
    {"code": "000300", "name": "沪深300", "reason": "CAPM分析的市场基准，代表A股大盘股"},
    {"code": "000905", "name": "中证500", "reason": "代表A股中小盘股票，与沪深300形成互补"},
]

print("开始下载指数数据...")
print("="*60)

for idx in index_list:
    code = idx["code"]
    name = idx["name"]
    
    print(f"\n正在下载: {name} ({code})...")
    
    success, msg = download_index_data(code, name, start_date, end_date, index_dir)
    
    if success:
        write_log(log_file, "SUCCESS", f"index_{code}", msg)
    else:
        write_log(log_file, "FAILED", f"index_{code}", msg)
    
    time.sleep(0.5)

print("\n" + "="*60)
print("指数数据下载完成！")

## 5. 下载宏观经济指标

In [ ]:
def download_cpi_data(save_dir):
    """
    下载CPI同比增速数据
    """
    try:
        # 获取CPI同比数据
        df = ak.macro_china_cpi_yearly()
        
        if df is None or len(df) == 0:
            return False, "No data returned"
        
        # 重命名列
        df.columns = ['date', 'cpi_yoy']
        
        # 筛选2020年以来的数据
        df['date'] = pd.to_datetime(df['date'])
        df = df[df['date'] >= '2020-01-01']
        df['date'] = df['date'].dt.strftime('%Y-%m')
        
        # 保存
        save_path = os.path.join(save_dir, "macro_cpi.csv")
        df.to_csv(save_path, index=False, encoding="utf-8")
        
        return True, f"shape=({len(df)}, {len(df.columns)})"
        
    except Exception as e:
        return False, str(e)

In [ ]:
def download_m2_data(save_dir):
    """
    下载M2同比增速数据
    """
    try:
        # 获取M2同比数据
        df = ak.macro_china_m2_yearly()
        
        if df is None or len(df) == 0:
            return False, "No data returned"
        
        # 重命名列
        df.columns = ['date', 'm2_yoy']
        
        # 筛选2020年以来的数据
        df['date'] = pd.to_datetime(df['date'])
        df = df[df['date'] >= '2020-01-01']
        df['date'] = df['date'].dt.strftime('%Y-%m')
        
        # 保存
        save_path = os.path.join(save_dir, "macro_m2.csv")
        df.to_csv(save_path, index=False, encoding="utf-8")
        
        return True, f"shape=({len(df)}, {len(df.columns)})"
        
    except Exception as e:
        return False, str(e)

In [ ]:
print("开始下载宏观经济数据...")
print("="*60)

# 下载CPI数据
print("\n正在下载: CPI同比增速...")
success, msg = download_cpi_data(macro_dir)
if success:
    write_log(log_file, "SUCCESS", "macro_cpi", msg)
else:
    write_log(log_file, "FAILED", "macro_cpi", msg)

time.sleep(0.5)

# 下载M2数据
print("\n正在下载: M2同比增速...")
success, msg = download_m2_data(macro_dir)
if success:
    write_log(log_file, "SUCCESS", "macro_m2", msg)
else:
    write_log(log_file, "FAILED", "macro_m2", msg)

print("\n" + "="*60)
print("宏观数据下载完成！")
print("\n宏观数据选择理由：")
print("- CPI同比增速：反映通货膨胀水平，影响货币政策和股票估值")
print("- M2同比增速：反映货币供应量变化，与股市流动性密切相关")

## 6. 下载财务指标数据

In [ ]:
def extract_finance_long_from_abstract(df, code):
    """将 stock_financial_abstract 宽表提取为长格式（每只股票最新可得5个年度）"""
    indicator_map = {
        "净资产收益率(ROE)": "ROE",
        "销售净利率": "净利润率"
    }

    # 仅保留“常用指标”分组，避免同名指标重复
    df = df[df['选项'].astype(str) == '常用指标'].copy()

    annual_cols = sorted(
        [c for c in df.columns if str(c).isdigit() and str(c).endswith('1231')],
        reverse=True
    )
    latest_annual_cols = annual_cols[:5]

    rows = []
    for raw_name, std_name in indicator_map.items():
        sub = df[df['指标'].astype(str) == raw_name]
        if sub.empty:
            continue

        s = sub.iloc[0]
        for col in latest_annual_cols:
            year = int(str(col)[:4])
            value = pd.to_numeric(s[col], errors='coerce')
            if pd.notna(value):
                rows.append({
                    'code': code,
                    'year': year,
                    'indicator': std_name,
                    'value': float(value)
                })

    return rows



In [ ]:
print("开始下载财务指标数据...")
print("="*60)

all_finance_data = []

for stock in stock_list:
    code = stock["code"]
    name = stock["name"]

    print(f"\n正在获取: {name} ({code})...")

    try:
        df = ak.stock_financial_abstract(symbol=code)

        if df is None or len(df) == 0:
            write_log(log_file, "FAILED", f"finance_{code}", "No data returned")
            continue

        rows = extract_finance_long_from_abstract(df, code)
        all_finance_data.extend(rows)

        write_log(log_file, "SUCCESS", f"finance_{code}", f"records={len(rows)}")
        print(f"  提取到 {len(rows)} 条（最新可得5年 × 2个指标）")

    except Exception as e:
        write_log(log_file, "FAILED", f"finance_{code}", str(e))
        print(f"  获取失败: {e}")

print("\n" + "="*60)
print("财务数据下载完成！")



In [ ]:
# 汇总并保存财务长格式数据
finance_df = pd.DataFrame(all_finance_data)

if len(finance_df) > 0:
    # 去重：同一股票同一年同指标只保留一条
    finance_df = finance_df.drop_duplicates(['code', 'year', 'indicator']).copy()
    finance_df = finance_df.sort_values(['code', 'year', 'indicator'])

    save_path = os.path.join(finance_dir, "finance_ratios.csv")
    finance_df.to_csv(save_path, index=False, encoding='utf-8')

    write_log(log_file, "SUCCESS", "finance_ratios", f"records={len(finance_df)}")

    print(f"财务数据已保存: {save_path}")
    print(f"总记录数: {len(finance_df)}")
    print(f"覆盖年度: {sorted(finance_df['year'].unique())}")
    print(f"指标类型: {finance_df['indicator'].unique().tolist()}")

    display(finance_df.head(20))
else:
    write_log(log_file, "FAILED", "finance_ratios", "No valid records extracted")
    print("未提取到有效财务数据")



## 7. 验证下载数据

In [ ]:
# 验证股票数据
print("股票数据验证：")
print("-" * 40)
for stock in stock_list:
    code = stock["code"]
    file_path = os.path.join(stock_dir, f"stock_{code}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"{stock['name']:8s} ({code}): {len(df)} 行, {len(df.columns)} 列")
    else:
        print(f"{stock['name']:8s} ({code}): 文件不存在")

In [ ]:
# 验证指数数据
print("\n指数数据验证：")
print("-" * 40)
for idx in index_list:
    code = idx["code"]
    file_path = os.path.join(index_dir, f"index_{code}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"{idx['name']:8s} ({code}): {len(df)} 行")
    else:
        print(f"{idx['name']:8s} ({code}): 文件不存在")

In [ ]:
# 验证宏观数据
print("\n宏观数据验证：")
print("-" * 40)
for macro_name in ["cpi", "m2"]:
    file_path = os.path.join(macro_dir, f"macro_{macro_name}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"{macro_name.upper():8s}: {len(df)} 行")
    else:
        print(f"{macro_name.upper():8s}: 文件不存在")

In [ ]:
# 查看下载日志
print("\n下载日志：")
print("-" * 60)
with open(log_file, "r", encoding="utf-8") as f:
    print(f.read())

## 8. 保存股票列表信息

In [ ]:
# 保存股票列表信息为CSV，便于后续使用
stock_info_df = pd.DataFrame(stock_list)
stock_info_df['select_reason'] = [
    "股份制银行龙头，零售业务领先",
    "国有大行代表，市值最大",
    "新能源汽车领军企业",
    "传统汽车制造龙头",
    "房地产开发龙头企业",
    "白酒行业龙头，高端消费代表",
    "浓香型白酒龙头",
    "能源行业龙头，油气开采",
    "通信设备龙头企业",
    "快递物流龙头企业"
]
stock_info_df.to_csv(os.path.join(data_dir, "stock_info.csv"), index=False, encoding="utf-8")
print("股票信息已保存")
display(stock_info_df)

In [ ]:
print("\n数据下载完成！")
print("="*60)
print("下一步：运行 02_clean.ipynb 进行数据清洗")